# Arabic ASR — Google Colab Training Notebook

**Run All** (`Runtime > Run all`) to train the model end-to-end without editing any code.

## Requirements before you start
1. Select a **GPU runtime** in Colab: *Runtime > Change runtime type > GPU (T4)*.
2. Upload the [Arabic Speech Corpus](https://en.arabicspeechcorpus.com/) to your Google Drive under the path shown in **Section 2** (or change the path to match your layout).

---
**Notebook sections**
| # | Section |
|---|--------|
| 1 | Install dependencies & mount Google Drive |
| 2 | Configuration — set all variables here |
| 3 | Build manifest & vocabulary |
| 4 | Train the model (with progress bars & loss plot) |
| 5 | Save model & vocabulary to Google Drive |
| 6 | Quick inference test |

---
## Section 1 — Install dependencies & mount Google Drive

In [ ]:
# Install required packages (TensorFlow is pre-installed on Colab GPU runtimes)
!pip install -q tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Google Drive mounted at /content/drive')

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {[g.name for g in gpus]}')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > GPU.')

---
## Section 2 — Configuration

**Edit only this cell** to match your Google Drive layout.

Expected Drive layout:
```
MyDrive/
  ASR_project/
    data/
      wav/   <- all .wav files
      lab/   <- all .lab files
```

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive'
PROJECT_DIR   = os.path.join(DRIVE_ROOT, 'ASR_project')   # project folder on Drive
WAV_DIR       = os.path.join(PROJECT_DIR, 'data', 'wav')  # folder containing .wav files
LAB_DIR       = os.path.join(PROJECT_DIR, 'data', 'lab')  # folder containing .lab files
MANIFEST_PATH = '/content/manifest.jsonl'                  # generated locally (faster I/O)
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints') # saved to Drive

# ── Audio settings ─────────────────────────────────────────────────────────
SAMPLE_RATE      = 16_000   # Hz
MAX_AUDIO_LENGTH = 10.0     # seconds — longer clips are skipped
N_FFT            = 512
HOP_LENGTH       = 160      # 10 ms
N_MELS           = 80

# ── Training hyper-parameters ──────────────────────────────────────────────
BATCH_SIZE        = 16      # increase if you have >=16 GB GPU VRAM
EPOCHS            = 50
LEARNING_RATE     = 3e-4
VALIDATION_SPLIT  = 0.1     # fraction of data used for validation

# ── LR scheduler ──────────────────────────────────────────────────────────
LR_PATIENCE = 5             # epochs without improvement before halving LR
LR_FACTOR   = 0.5
LR_MIN      = 1e-6

# ── Gradient clipping ──────────────────────────────────────────────────────
GRAD_CLIP_NORM = 5.0

# ── Verify data dirs exist ─────────────────────────────────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
assert os.path.isdir(WAV_DIR), f'WAV directory not found: {WAV_DIR}'
assert os.path.isdir(LAB_DIR), f'LAB directory not found: {LAB_DIR}'
print('Configuration OK')
print(f'  WAV dir       : {WAV_DIR}')
print(f'  LAB dir       : {LAB_DIR}')
print(f'  Checkpoint dir: {CHECKPOINT_DIR}')

---
## Section 3 — Build manifest & vocabulary

In [ ]:
import json
import glob
import random
import numpy as np

# ── Read .lab transcription files ─────────────────────────────────────────
def read_lab_files(lab_dir):
    transcripts = {}
    for path in sorted(glob.glob(os.path.join(lab_dir, '*.lab'))):
        utt_id = os.path.splitext(os.path.basename(path))[0]
        with open(path, 'r', encoding='utf-8') as f:
            text = f.read().strip()
        if text:
            transcripts[utt_id] = text
    return transcripts

# ── Build JSONL manifest ───────────────────────────────────────────────────
def build_manifest(wav_dir, lab_dir, output_path):
    transcripts = read_lab_files(lab_dir)
    entries = []
    for wav_path in sorted(glob.glob(os.path.join(wav_dir, '*.wav'))):
        utt_id = os.path.splitext(os.path.basename(wav_path))[0]
        text   = transcripts.get(utt_id)
        if text is None:
            continue
        # Rough duration via file size (PCM 16-bit mono)
        duration = os.path.getsize(wav_path) / (SAMPLE_RATE * 2)
        if duration > MAX_AUDIO_LENGTH:
            continue
        entries.append({'wav': wav_path, 'text': text, 'duration': round(duration, 3)})
    with open(output_path, 'w', encoding='utf-8') as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + '\n')
    print(f'Manifest saved: {output_path}  ({len(entries)} utterances)')
    return entries

entries = build_manifest(WAV_DIR, LAB_DIR, MANIFEST_PATH)

# ── Shuffle and split ─────────────────────────────────────────────────────
random.seed(42)
random.shuffle(entries)
num_val       = int(len(entries) * VALIDATION_SPLIT)
val_entries   = entries[:num_val]
train_entries = entries[num_val:]
print(f'Training: {len(train_entries)}   Validation: {len(val_entries)}')

In [ ]:
# ── Build character vocabulary ────────────────────────────────────────────
def build_vocab(texts):
    chars      = sorted(set(''.join(texts)))
    char_to_id = {c: i + 1 for i, c in enumerate(chars)}
    char_to_id['<blank>'] = 0          # index 0 reserved for CTC blank
    id_to_char = {v: k for k, v in char_to_id.items()}
    return char_to_id, id_to_char

def encode_text(text, char_to_id):
    return [char_to_id[c] for c in text if c in char_to_id]

texts              = [e['text'] for e in entries]
char_to_id, id_to_char = build_vocab(texts)
VOCAB_SIZE         = len(char_to_id)
print(f'Vocabulary size: {VOCAB_SIZE} characters (including blank)')

In [ ]:
# ── Audio helpers ─────────────────────────────────────────────────────────
def load_wav(path):
    raw   = tf.io.read_file(path)
    audio, _ = tf.audio.decode_wav(raw, desired_channels=1)
    return tf.squeeze(audio, axis=-1)

def wav_to_mel(audio):
    stft      = tf.signal.stft(audio, frame_length=N_FFT, frame_step=HOP_LENGTH)
    magnitude = tf.abs(stft)
    num_bins  = magnitude.shape[-1] or (N_FFT // 2 + 1)
    mel_w     = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=N_MELS, num_spectrogram_bins=num_bins,
        sample_rate=SAMPLE_RATE, lower_edge_hertz=80.0, upper_edge_hertz=7600.0)
    log_mel   = tf.math.log(tf.matmul(magnitude, mel_w) + 1e-6)
    return log_mel   # (time, N_MELS)

# ── tf.data pipeline ──────────────────────────────────────────────────────
def create_dataset(entries_list, char_to_id_map, batch_size):
    wav_paths = [e['wav']  for e in entries_list]
    labels    = [encode_text(e['text'], char_to_id_map) for e in entries_list]

    def generator():
        for wp, lab in zip(wav_paths, labels):
            audio = load_wav(wp)
            mel   = wav_to_mel(audio)
            yield mel, np.array(lab, dtype=np.int32)

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, N_MELS), dtype=tf.float32),
            tf.TensorSpec(shape=(None,),        dtype=tf.int32),
        ),
    )
    ds = ds.padded_batch(
        batch_size,
        padded_shapes=([None, N_MELS], [None]),
        padding_values=(0.0, 0),
    )
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = create_dataset(train_entries, char_to_id, BATCH_SIZE)
val_ds   = create_dataset(val_entries,   char_to_id, BATCH_SIZE)
print('Datasets created')

---
## Section 4 — Train the model

In [ ]:
from tensorflow.keras import layers, Model

def build_model(n_mels, vocab_size):
    inputs = layers.Input(shape=(None, n_mels), name='audio_input')
    # conv1: stride=2  =>  T_out = ceil(T_in / 2)
    x = layers.Conv1D(128, kernel_size=11, strides=2, padding='same', activation='relu', name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    # conv2: stride=1  =>  no additional downsampling
    x = layers.Conv1D(128, kernel_size=11, strides=1, padding='same', activation='relu', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Dropout(0.2, name='drop1')(x)
    x = layers.Bidirectional(layers.GRU(256, return_sequences=True), name='bigru1')(x)
    x = layers.Dropout(0.2, name='drop2')(x)
    x = layers.Bidirectional(layers.GRU(256, return_sequences=True), name='bigru2')(x)
    x = layers.Dropout(0.2, name='drop3')(x)
    outputs = layers.Dense(vocab_size, activation=None, name='output')(x)
    return Model(inputs, outputs, name='ASR_CTC')

model = build_model(n_mels=N_MELS, vocab_size=VOCAB_SIZE)
model.summary()

In [ ]:
# ── CTC loss ──────────────────────────────────────────────────────────────
def ctc_loss_fn(y_true, y_pred, input_lengths, label_lengths):
    return tf.reduce_mean(
        tf.nn.ctc_loss(
            labels=tf.cast(y_true, tf.int32),
            logits=y_pred,
            label_length=tf.cast(label_lengths, tf.int32),
            logit_length=tf.cast(input_lengths, tf.int32),
            logits_time_major=False,
            blank_index=0,
        )
    )

# ── Input-length helper (accounts for both Conv strides) ──────────────────
def compute_input_lengths(spectrograms):
    mask    = tf.reduce_sum(tf.abs(spectrograms), axis=-1)          # (batch, time)
    lengths = tf.reduce_sum(tf.cast(mask != 0, tf.int32), axis=-1)  # (batch,)
    return (lengths + 1) // 2   # combined stride factor = 2

def compute_label_lengths(labels):
    return tf.reduce_sum(tf.cast(labels != 0, tf.int32), axis=-1)

# ── LR scheduler ──────────────────────────────────────────────────────────
class ReduceLROnPlateau:
    def __init__(self, optimizer, patience=5, factor=0.5, min_lr=1e-6):
        self.optimizer = optimizer
        self.patience  = patience
        self.factor    = factor
        self.min_lr    = min_lr
        self._best     = float('inf')
        self._wait     = 0

    def step(self, val_loss):
        if val_loss < self._best:
            self._best = val_loss
            self._wait = 0
        else:
            self._wait += 1
            if self._wait >= self.patience:
                old_lr = float(self.optimizer.learning_rate)
                new_lr = max(old_lr * self.factor, self.min_lr)
                self.optimizer.learning_rate.assign(new_lr)
                self._wait = 0
                print(f'  [LR] {old_lr:.2e} -> {new_lr:.2e}')
        return float(self.optimizer.learning_rate)

optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
scheduler = ReduceLROnPlateau(optimizer, patience=LR_PATIENCE, factor=LR_FACTOR, min_lr=LR_MIN)
print('Optimizer and scheduler ready')

In [ ]:
import time
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

train_losses = []
val_losses   = []
best_val_loss = float('inf')
best_model_path = os.path.join(CHECKPOINT_DIR, 'model_best.keras')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # Training
    train_loss = 0.0
    n_batches  = 0
    for spectrograms, labels in tqdm(train_ds, desc=f'Epoch {epoch}/{EPOCHS} [train]', leave=False):
        input_lengths = compute_input_lengths(spectrograms)
        label_lengths = compute_label_lengths(labels)
        with tf.GradientTape() as tape:
            y_pred = model(spectrograms, training=True)
            loss   = ctc_loss_fn(labels, y_pred, input_lengths, label_lengths)
        grads, _ = tf.clip_by_global_norm(
            tape.gradient(loss, model.trainable_variables), GRAD_CLIP_NORM)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        train_loss += loss.numpy()
        n_batches  += 1
    avg_train = train_loss / max(n_batches, 1)

    # Validation
    val_loss  = 0.0
    n_val     = 0
    for spectrograms, labels in val_ds:
        input_lengths = compute_input_lengths(spectrograms)
        label_lengths = compute_label_lengths(labels)
        y_pred    = model(spectrograms, training=False)
        val_loss += ctc_loss_fn(labels, y_pred, input_lengths, label_lengths).numpy()
        n_val    += 1
    avg_val = val_loss / max(n_val, 1)

    current_lr = scheduler.step(avg_val)
    elapsed    = time.time() - t0

    train_losses.append(avg_train)
    val_losses.append(avg_val)

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'train={avg_train:.4f}  val={avg_val:.4f}  '
          f'lr={current_lr:.2e}  ({elapsed:.0f}s)')

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save(best_model_path)
        print(f'  New best — saved to {best_model_path}')

    if epoch % 10 == 0:
        ckpt = os.path.join(CHECKPOINT_DIR, f'model_epoch_{epoch}.keras')
        model.save(ckpt)
        print(f'  Checkpoint: {ckpt}')

print('\nTraining complete.')

In [ ]:
# Plot training and validation loss curves
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train loss')
plt.plot(range(1, len(val_losses)   + 1), val_losses,   label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('CTC Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'loss_curve.png'), dpi=120)
plt.show()
print('Loss curve saved to', os.path.join(CHECKPOINT_DIR, 'loss_curve.png'))

---
## Section 5 — Save final model & vocabulary to Google Drive

In [ ]:
# Save the final model
final_model_path = os.path.join(CHECKPOINT_DIR, 'model_final.keras')
model.save(final_model_path)
print(f'Final model saved : {final_model_path}')

# Save vocabulary
vocab_path = os.path.join(CHECKPOINT_DIR, 'vocab.json')
with open(vocab_path, 'w', encoding='utf-8') as f:
    json.dump(char_to_id, f, ensure_ascii=False, indent=2)
print(f'Vocabulary saved  : {vocab_path}')

print('\nAll artifacts saved to Google Drive at:', CHECKPOINT_DIR)

---
## Section 6 — Quick inference test

Loads the saved model and runs a single prediction on the first WAV file in the corpus.

In [ ]:
# Greedy CTC decode
def ctc_greedy_decode(y_pred, id_to_char_map):
    indices = tf.argmax(y_pred, axis=-1).numpy()
    results = []
    for seq in indices:
        chars, prev = [], -1
        for idx in seq:
            if idx != 0 and idx != prev:
                chars.append(id_to_char_map.get(int(idx), ''))
            prev = idx
        results.append(''.join(chars))
    return results

# ── Load model & vocab ────────────────────────────────────────────────────
saved_model = tf.keras.models.load_model(final_model_path)
with open(vocab_path, 'r', encoding='utf-8') as f:
    saved_char_to_id = json.load(f)
saved_id_to_char = {int(v): k for k, v in saved_char_to_id.items()}
print(f'Model loaded  : {final_model_path}')
print(f'Vocab size    : {len(saved_char_to_id)}')

# ── Pick a sample file ────────────────────────────────────────────────────
sample_wav = sorted(glob.glob(os.path.join(WAV_DIR, '*.wav')))[0]
print(f'Sample file   : {sample_wav}')

# ── Get ground-truth label ────────────────────────────────────────────────
sample_id  = os.path.splitext(os.path.basename(sample_wav))[0]
lab_path   = os.path.join(LAB_DIR, sample_id + '.lab')
ground_truth = open(lab_path, encoding='utf-8').read().strip() if os.path.exists(lab_path) else '(no label)'

# ── Run inference ─────────────────────────────────────────────────────────
audio    = load_wav(sample_wav)
mel      = wav_to_mel(audio)
mel_batch = tf.expand_dims(mel, axis=0)
y_pred   = saved_model.predict(mel_batch, verbose=0)
decoded  = ctc_greedy_decode(y_pred, saved_id_to_char)

print('\n--- Inference result ---')
print(f'Ground truth : {ground_truth}')
print(f'Prediction   : {decoded[0]}')